# Liveness 05 · Evaluation (ISO/IEC 30107-3)

Reports **APCER** (worst attack type), **BPCER** and **ACER** for: device passive model, server ViT, and the
fused decision; intra-dataset and **cross-dataset** (MSU + Replay unseen). Spec target: **APCER ≤ 2 %, BPCER ≤ 5 %**.
The operating threshold is chosen at BPCER = 5 % and stored per station in `tenant_ai_models.config`.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Liveness
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import numpy as np, pandas as pd
from Common_Utils.Metrics_Utils import apcer_bpcer, threshold_at_bpcer

# scores.csv columns: score_device, score_server, label (1 live / 0 spoof), attack_type, source
path = DATA / "Liveness" / "Scores.csv"
if path.exists():
    df = pd.read_csv(path)
    df["score_fused"] = 0.4 * df.score_device + 0.6 * df.score_server
    for col in ["score_device", "score_server", "score_fused"]:
        thr = threshold_at_bpcer(df[col].values, df.label.values, 0.05)
        print(col, round(thr, 3), apcer_bpcer(df[col].values, df.label.values, df.attack_type.values, thr))
else:
    print("Run the models on the test split and save", path)